# 01 · Keşif (EDA) — Biohub Cell Tracking · *kapsamlı*

Zebra balığı embriyosunda 3B+zaman hücre takibi. Bu defter veriyi **çalışma anında
keşfeder** ve bol görsel üretir:

1. Dataset geneli özet (tüm train örnekleri)
2. Görüntü: boyut, ölçek, yoğunluk, derinlik zayıflaması, dilim/zaman montajları
3. GEFF ground-truth grafı: seyreklik, soylar, bölünmeler
4. **Koordinat birimi** (µm mi voxel mi) — kesin tespit
5. Hareket analizi (eksen bazlı, hız, zaman)
6. Çekirdek görünümü (yama montajı, GT vs rastgele yoğunluk)
7. Soy (lineage) yörüngesi

> Görseller hem inline gösterilir hem `/kaggle/working/figures/`'a kaydedilir →
> Kaggle **Output**'tan indirip repoda `outputs/figures/`'a alacağız.

## 0 · Kurulum

In [ ]:
# Kaggle'da numpy/scipy/matplotlib/pandas/zarr HAZIR. Kurulum YAPMIYORUZ
# (tracksdata kurulumu numpy ABI'sini bozuyor). GEFF'i ham zarr okuyacagiz.
import importlib, subprocess, sys
def has(pkg):
    try: importlib.import_module(pkg); return True
    except Exception: return False
if not has("zarr"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zarr"], check=False)
print("zarr:", has("zarr"), "| geff:", has("geff"), "| tracksdata:", has("tracksdata"))

In [ ]:
import os, json, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import matplotlib as mpl
warnings.filterwarnings("ignore")
mpl.rcParams["figure.facecolor"] = "white"
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.alpha"] = 0.25

FIG_DIR = Path("/kaggle/working/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
def savefig(name):
    plt.savefig(FIG_DIR / name, dpi=120, bbox_inches="tight")
    plt.show()

SCALE_ZYX = (1.625, 0.40625, 0.40625)   # YEDEK; 3. bolumde dosyadan okunacak
COORDS_UM = None                         # koordinat birimi 4. bolumde belirlenecek
print("Hazir. Yedek olcek (Z,Y,X) um/px:", SCALE_ZYX)

## 0b · Yardımcı okuyucular (OME-Zarr görüntü + ham GEFF grafı)

In [ ]:
def open_image(zpath):
    # OME-Zarr ac; en yuksek cozunurluklu diziyi ve olcegi (Z,Y,X um) dondur.
    node = zarr.open(str(zpath), mode="r")
    attrs = dict(node.attrs) if hasattr(node, "attrs") else {}
    arr, scale = None, None
    ms = attrs.get("multiscales")
    if ms is None and isinstance(attrs.get("ome"), dict):
        ms = attrs["ome"].get("multiscales")     # OME-NGFF 0.5 (Zarr v3)
    if ms:
        try:
            ds0 = ms[0]["datasets"][0]
            arr = node[ds0["path"]]
            for tf in ds0.get("coordinateTransformations", []):
                if tf.get("type") == "scale":
                    s = [float(v) for v in tf["scale"]]
                    scale = tuple(s[-3:])         # (Z,Y,X)
        except Exception as e:
            print("[uyari] multiscale cozulemedi:", e)
    if arr is None:
        keys = list(node.keys()) if hasattr(node, "keys") else []
        arr = node["0"] if "0" in keys else node
    return arr, attrs, scale

def load_geff_raw(gpath):
    # GEFF'i ham zarr olarak oku -> node ids + props + edges + axes meta.
    g = zarr.open(str(gpath), mode="r")
    ra = dict(g.attrs)
    axes = ra["axes"] if isinstance(ra.get("axes"), list) else None
    for key in ("geff", "geff_metadata"):
        v = ra.get(key)
        if isinstance(v, dict) and v.get("axes"): axes = v["axes"]
    nodes = g["nodes"]; node_ids = np.asarray(nodes["ids"])
    props = {}
    if "props" in nodes:
        for pn in list(nodes["props"].keys()):
            try: props[pn] = np.asarray(nodes["props"][pn]["values"])
            except Exception: pass
    edges = np.asarray(g["edges"]["ids"])
    return {"node_ids": node_ids, "props": props, "prop_names": list(props.keys()),
            "edges": edges, "axes": axes, "root_attrs": ra}

def nodes_to_frame(gd):
    props = gd["props"]; axes = gd.get("axes") or []
    tn = zn = yn = xn = None
    for a in axes:
        if not isinstance(a, dict): continue
        nm = a.get("name"); tp = str(a.get("type", "")).lower()
        if tp in ("time", "t") or nm in ("t", "time", "frame"): tn = nm
        elif nm in ("z", "Z"): zn = nm
        elif nm in ("y", "Y"): yn = nm
        elif nm in ("x", "X"): xn = nm
    def pick(name, *c):
        if name and name in props: return props[name]
        for k in c:
            if k in props: return props[k]
        return None
    cols = {"id": gd["node_ids"]}
    for key, val in [("t", pick(tn,"t","time","frame","T")), ("z", pick(zn,"z","Z")),
                     ("y", pick(yn,"y","Y")), ("x", pick(xn,"x","X"))]:
        if val is not None: cols[key] = val
    return pd.DataFrame(cols)

def uf_count_tracks(node_ids, edges):
    # zayif bagli bilesen (soy) sayisi
    parent = {int(i): int(i) for i in node_ids}
    def find(a):
        while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for s, d in edges:
        s, d = int(s), int(d)
        if s in parent and d in parent: parent[find(s)] = find(d)
    return len({find(int(i)) for i in node_ids})
print("yardimcilar hazir")

## 1 · Girdi ağacı ve örnek listesi

In [ ]:
INPUT = Path("/kaggle/input")
def find_root():
    if not INPUT.exists(): return None
    stack = [(INPUT, 0)]
    while stack:
        base, d = stack.pop()
        try:
            if (base/"train").is_dir() and (base/"test").is_dir(): return base
        except Exception: pass
        if d < 4:
            for c in sorted(base.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr", ".geff")):
                    stack.append((c, d+1))
    return None
ROOT = find_root(); print("Kok:", ROOT)
if ROOT:
    print("Icerik:", [e.name for e in sorted(ROOT.iterdir())])

def split_names(split):
    d = ROOT/split if ROOT else None
    if not d or not d.is_dir(): d = ROOT
    zarrs = sorted(p.stem for p in d.glob("*.zarr")) if d else []
    return d, zarrs

train_dir, train_names = split_names("train")
test_dir,  test_names  = split_names("test")
print(f"train ornek: {len(train_names)} | test ornek: {len(test_names)}")
print("ilk train:", train_names[:5])

## 2 · Dataset geneli özet (tüm train `.geff` + görüntü boyutları)
Her örnek için node/edge/soy/bölünme sayıları ve görüntü boyutu.

In [ ]:
rows = []
for nm in train_names:
    gp = train_dir/(nm + ".geff")
    try:
        gd = load_geff_raw(gp); ndf = nodes_to_frame(gd); E = gd["edges"]
        outdeg = Counter(int(s) for s, _ in E)
        rows.append(dict(sample=nm, n_nodes=len(gd["node_ids"]), n_edges=len(E),
                         n_tracks=uf_count_tracks(gd["node_ids"], E),
                         n_div=sum(1 for v in outdeg.values() if v >= 2),
                         t_min=int(ndf.t.min()) if "t" in ndf else -1,
                         t_max=int(ndf.t.max()) if "t" in ndf else -1))
    except Exception as e:
        print("[skip geff]", nm, e)
summ = pd.DataFrame(rows)
print("Ozet tablo (ilk satirlar):"); print(summ.head(10).to_string(index=False))
print("\nToplamlar:", dict(nodes=int(summ.n_nodes.sum()), edges=int(summ.n_edges.sum()),
      tracks=int(summ.n_tracks.sum()), div=int(summ.n_div.sum())))
print("\nIstatistik:"); print(summ[["n_nodes","n_edges","n_tracks","n_div"]].describe().round(1).to_string())

In [ ]:
# Goruntu boyutlari (yalnizca metadata; piksel yuklemeden)
srows = []
for nm in train_names:
    try:
        arr, _, sc = open_image(train_dir/(nm + ".zarr"))
        sh = arr.shape
        srows.append(dict(sample=nm, T=sh[0], Z=sh[1], Y=sh[2], X=sh[3],
                          scale=str(sc), dtype=str(arr.dtype)))
    except Exception as e:
        print("[skip img]", nm, e)
shapes = pd.DataFrame(srows)
print(shapes.head(10).to_string(index=False))
print("\nBoyut araliklari:")
for c in ["T","Z","Y","X"]:
    if c in shapes: print(f"  {c}: min={shapes[c].min()} max={shapes[c].max()}")
print("olcekler:", shapes["scale"].unique()[:5] if "scale" in shapes else None)
print("dtype:", shapes["dtype"].unique() if "dtype" in shapes else None)

### 2b · Dağılımlar (örnekler arası)

In [ ]:
if len(summ):
    fig, ax = plt.subplots(2, 3, figsize=(16, 8))
    ax = ax.ravel()
    ax[0].hist(summ.n_nodes, bins=20, color="#4C78A8"); ax[0].set_title("node / örnek")
    ax[1].hist(summ.n_edges, bins=20, color="#4C78A8"); ax[1].set_title("edge / örnek")
    ax[2].hist(summ.n_tracks, bins=20, color="#F58518"); ax[2].set_title("soy (track) / örnek")
    ax[3].hist(summ.n_div, bins=20, color="#E45756"); ax[3].set_title("bölünme / örnek")
    if len(shapes):
        ax[4].hist(shapes["T"], bins=20, color="#72B7B2"); ax[4].set_title("T (zaman) / örnek")
        ax[5].scatter(shapes["Y"], shapes["X"], s=20); ax[5].set_title("Y vs X (piksel)")
        ax[5].set_xlabel("Y"); ax[5].set_ylabel("X")
    fig.suptitle("Dataset geneli dağılımlar", fontsize=14)
    savefig("02_dataset_distributions.png")

## 3 · Bir görüntüyü derinlemesine incele

In [ ]:
SAMPLE = train_names[0]
img, img_attrs, img_scale = open_image(train_dir/(SAMPLE + ".zarr"))
print("Örnek:", SAMPLE)
print("shape (T,Z,Y,X):", img.shape, "| dtype:", img.dtype, "| chunks:", getattr(img,"chunks",None))
print("attrs multiscales scale (Z,Y,X):", img_scale)
if img_scale and len(img_scale) == 3:
    SCALE_ZYX = tuple(float(v) for v in img_scale)
    print(">> Kullanilacak olcek (dosyadan):", SCALE_ZYX)
else:
    print(">> Olcek okunamadi; YEDEK:", SCALE_ZYX)
T, Z, Y, X = img.shape
t_mid = T // 2
vol = np.asarray(img[t_mid]).astype(np.float32)   # (Z,Y,X)
print(f"t={t_mid} hacim:", vol.shape, "| min/max:", float(vol.min()), float(vol.max()))

### 3a · Yoğunluk dağılımı ve derinlik (Z) zayıflaması

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
# 1) yogunluk histogrami (log)
ax[0].hist(vol.ravel()[::11], bins=120, color="#4C78A8"); ax[0].set_yscale("log")
ax[0].set_title("Yoğunluk histogramı (log)"); ax[0].set_xlabel("intensite")
# 2) Z-derinligi boyunca ortalama yogunluk (light-sheet zayiflamasi)
z_mean = vol.reshape(Z, -1).mean(axis=1)
ax[1].plot(np.arange(Z) * SCALE_ZYX[0], z_mean, "-o", ms=3)
ax[1].set_title("Z-derinliği boyunca ort. yoğunluk"); ax[1].set_xlabel("derinlik (µm)")
ax[1].set_ylabel("ort. intensite")
# 3) zaman boyunca ortalama yogunluk (birkac t ornekle)
ts = np.linspace(0, T-1, min(T, 20)).astype(int)
t_mean = [float(np.asarray(img[t]).mean()) for t in ts]
ax[2].plot(ts, t_mean, "-o", ms=3, color="#F58518")
ax[2].set_title("Zaman boyunca ort. yoğunluk"); ax[2].set_xlabel("t")
fig.suptitle(f"{SAMPLE} — yoğunluk profilleri"); savefig("03_intensity_profiles.png")

### 3b · Maksimum-yoğunluk projeksiyonları (anizotropi görünür)

In [ ]:
mxy, mxz, myz = vol.max(0), vol.max(1), vol.max(2)
fig, ax = plt.subplots(1, 3, figsize=(17, 6))
for a, m, ttl in zip(ax, [mxy, mxz, myz], ["XY (Z-MIP)", "XZ (Y-MIP)", "YZ (X-MIP)"]):
    a.imshow(m, cmap="magma"); a.set_title(ttl); a.axis("off")
fig.suptitle(f"{SAMPLE} — t={t_mid}"); savefig("03b_mip_xyz.png")

### 3c · Z-dilim montajı (tek zaman) — hacmin derinlik boyunca kesitleri

In [ ]:
nz = min(Z, 12); zs = np.linspace(0, Z-1, nz).astype(int)
cols = 4; rowsn = int(np.ceil(nz / cols))
fig, ax = plt.subplots(rowsn, cols, figsize=(4*cols, 4*rowsn))
ax = np.array(ax).ravel()
vmax = np.percentile(vol, 99.5)
for i, z in enumerate(zs):
    ax[i].imshow(vol[z], cmap="gray", vmax=vmax); ax[i].set_title(f"z={z}"); ax[i].axis("off")
for j in range(nz, len(ax)): ax[j].axis("off")
fig.suptitle(f"{SAMPLE} t={t_mid} — Z dilimleri"); savefig("03c_zslices.png")

### 3d · Zaman montajı (XY-MIP birkaç zaman noktası)

In [ ]:
nt = min(T, 8); tt = np.linspace(0, T-1, nt).astype(int)
cols = 4; rowsn = int(np.ceil(nt / cols))
fig, ax = plt.subplots(rowsn, cols, figsize=(4*cols, 4*rowsn)); ax = np.array(ax).ravel()
for i, t in enumerate(tt):
    m = np.asarray(img[t]).astype(np.float32).max(0)
    ax[i].imshow(m, cmap="gray", vmax=np.percentile(m, 99.5))
    ax[i].set_title(f"t={t}"); ax[i].axis("off")
for j in range(nt, len(ax)): ax[j].axis("off")
fig.suptitle(f"{SAMPLE} — zaman boyunca XY-MIP"); savefig("03d_timemontage.png")

## 4 · Ground-truth grafı (bu örnek)

In [ ]:
gd = load_geff_raw(train_dir/(SAMPLE + ".geff"))
ndf = nodes_to_frame(gd); E = gd["edges"]
print("node prop adlari:", gd["prop_names"])
print("axes meta:", gd["axes"])
print("#nodes:", len(gd["node_ids"]), "| #edges:", len(E))
print("kolonlar:", list(ndf.columns))
print(ndf.head())
outdeg = Counter(int(s) for s, _ in E); indeg = Counter(int(d) for _, d in E)
n_div = sum(1 for v in outdeg.values() if v >= 2)
n_start = sum(1 for i in gd["node_ids"] if indeg.get(int(i), 0) == 0)
n_end   = sum(1 for i in gd["node_ids"] if outdeg.get(int(i), 0) == 0)
n_tracks = uf_count_tracks(gd["node_ids"], E)
print(f"bolunme:{n_div} | baslangic:{n_start} | bitis:{n_end} | soy(bilesen):{n_tracks}")

### 4a · Seyreklik, soy uzunlukları, bölünme zamanı

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
# zaman basina node
if "t" in ndf:
    per_t = ndf.groupby("t").size()
    ax[0].bar(per_t.index, per_t.values, color="#4C78A8")
    ax[0].set_title(f"node/zaman (ort {per_t.mean():.1f})"); ax[0].set_xlabel("t")
# soy uzunluklari (union-find bilesen boyutu)
parent = {int(i): int(i) for i in gd["node_ids"]}
def find(a):
    while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
    return a
for s, d in E:
    s, d = int(s), int(d)
    if s in parent and d in parent: parent[find(s)] = find(d)
comp = Counter(find(int(i)) for i in gd["node_ids"])
ax[1].hist(list(comp.values()), bins=30, color="#F58518")
ax[1].set_title("soy uzunluğu (node sayısı)"); ax[1].set_xlabel("soy başına node")
# bolunme zamani
div_nodes = [n for n, v in outdeg.items() if v >= 2]
if div_nodes and "t" in ndf:
    id2t = dict(zip(ndf.id, ndf.t))
    dts = [id2t[n] for n in div_nodes if n in id2t]
    ax[2].hist(dts, bins=20, color="#E45756"); ax[2].set_title(f"bölünme zamanı (n={len(dts)})")
    ax[2].set_xlabel("t")
else:
    ax[2].text(0.5, 0.5, "bu örnekte bölünme yok", ha="center"); ax[2].set_title("bölünme zamanı")
savefig("04_graph_stats.png")

## 5 · Koordinat birimi: µm mi voxel mi? *(linking için kritik)*
Koordinat aralığını görüntü boyutuyla ve `scale` ile kıyaslarız.

In [ ]:
print("Görüntü boyutu (Z,Y,X):", (Z, Y, X))
for c, dim, sc in [("z", Z, SCALE_ZYX[0]), ("y", Y, SCALE_ZYX[1]), ("x", X, SCALE_ZYX[2])]:
    if c in ndf:
        cmax = float(ndf[c].max()); cmin = float(ndf[c].min())
        r_vox = cmax / dim
        r_um  = cmax / (dim * sc)
        print(f"{c}: min={cmin:.2f} max={cmax:.2f} | dim={dim} scale={sc} "
              f"| max/dim={r_vox:.3f}  max/(dim*scale)={r_um:.3f}")
# Karar: max/dim ~1 -> voxel; max/(dim*scale) ~1 -> um
xmax = float(ndf["x"].max()) if "x" in ndf else 0
COORDS_UM = abs(xmax/(X*SCALE_ZYX[2]) - 1) < abs(xmax/X - 1)
print("\n>> COORDS_UM (koordinatlar mikrometre mi?):", COORDS_UM)
print(">> Bu karar linking mesafe hesabinda kullanilacak.")

### 5a · GT node'larının uzamsal dağılımı (koordinat uzayında)

In [ ]:
if set(["x","y","z"]).issubset(ndf.columns):
    fig, ax = plt.subplots(1, 3, figsize=(17, 5))
    sc = ax[0].scatter(ndf.x, ndf.y, c=ndf.t if "t" in ndf else None, cmap="viridis", s=10)
    ax[0].set_title("XY (renk=t)"); ax[0].set_xlabel("x"); ax[0].set_ylabel("y"); ax[0].invert_yaxis()
    ax[1].scatter(ndf.x, ndf.z, c=ndf.t if "t" in ndf else None, cmap="viridis", s=10)
    ax[1].set_title("XZ"); ax[1].set_xlabel("x"); ax[1].set_ylabel("z")
    ax[2].scatter(ndf.y, ndf.z, c=ndf.t if "t" in ndf else None, cmap="viridis", s=10)
    ax[2].set_title("YZ"); ax[2].set_xlabel("y"); ax[2].set_ylabel("z")
    plt.colorbar(sc, ax=ax, fraction=0.02, label="t")
    fig.suptitle(f"{SAMPLE} — GT node uzamsal dağılımı"); savefig("05_gt_spatial.png")

## 6 · Hareket analizi — eksen bazlı yer değiştirme, hız

In [ ]:
P = ndf.set_index("id")[["z","y","x"]] if set(["z","y","x"]).issubset(ndf.columns) else None
sz, sy, sx = SCALE_ZYX
def to_um(dzyx):
    dzyx = np.asarray(dzyx, float)
    return dzyx if COORDS_UM else dzyx * np.array([sz, sy, sx])
disp_um, dz_um, dy_um, dx_um, src_t = [], [], [], [], []
id2t = dict(zip(ndf.id, ndf.t)) if "t" in ndf else {}
if P is not None:
    for s, d in E:
        s, d = int(s), int(d)
        if s in P.index and d in P.index:
            dd = to_um(P.loc[d].values - P.loc[s].values)
            dz_um.append(dd[0]); dy_um.append(dd[1]); dx_um.append(dd[2])
            disp_um.append(float(np.sqrt((dd**2).sum())))
            src_t.append(id2t.get(s, np.nan))
disp_um = np.array(disp_um)
print("edge sayisi (mesafeli):", len(disp_um))
if len(disp_um):
    print("yer degistirme um -> medyan:%.2f  ort:%.2f  95p:%.2f  max:%.2f" % (
        np.median(disp_um), disp_um.mean(), np.percentile(disp_um,95), disp_um.max()))

In [ ]:
if len(disp_um):
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    ax[0].hist(disp_um, bins=50, color="#4C78A8")
    ax[0].axvline(7, color="r", ls="--", label="7 µm tolerans")
    ax[0].set_title("|yer değiştirme| (µm)"); ax[0].set_xlabel("µm"); ax[0].legend()
    # eksen bazli
    ax[1].hist(np.abs(dz_um), bins=40, alpha=0.6, label="|dz|")
    ax[1].hist(np.abs(dy_um), bins=40, alpha=0.6, label="|dy|")
    ax[1].hist(np.abs(dx_um), bins=40, alpha=0.6, label="|dx|")
    ax[1].set_title("eksen bazlı |Δ| (µm)"); ax[1].set_xlabel("µm"); ax[1].legend()
    # hiz vs zaman
    if np.isfinite(src_t).any():
        order = np.argsort(src_t)
        ax[2].scatter(np.array(src_t)[order], disp_um[order], s=8, alpha=0.5)
        ax[2].set_title("hız (µm/kare) vs t"); ax[2].set_xlabel("t"); ax[2].set_ylabel("µm/kare")
    fig.suptitle(f"{SAMPLE} — hareket"); savefig("06_motion.png")

## 7 · Çekirdek görünümü — GT konumları gerçekten çekirdek mi?

In [ ]:
# Birkac zaman noktasini yukle (yeniden kullanmak icin), GT node'lariyla eslestir.
N_T = min(6, T)
t_with_nodes = sorted(set(ndf.t.astype(int))) if "t" in ndf else list(range(T))
pick_t = [int(x) for x in np.linspace(0, len(t_with_nodes)-1, min(N_T, len(t_with_nodes))).astype(int)]
pick_t = sorted(set(t_with_nodes[i] for i in pick_t))
vols = {t: np.asarray(img[t]).astype(np.float32) for t in pick_t}
print("yuklenen zaman noktalari:", pick_t)

# GT node cevresinden yamalar (z-dilimi + kare kesit)
R = 24
patches = []
for t in pick_t:
    sub = ndf[ndf.t == t] if "t" in ndf else ndf.iloc[:0]
    for _, r in sub.iterrows():
        z, y, x = int(round(r.z)), int(round(r.y)), int(round(r.x))
        v = vols[t]
        y0, y1 = max(0, y-R), min(Y, y+R); x0, x1 = max(0, x-R), min(X, x+R)
        z = min(max(z, 0), v.shape[0]-1)
        patches.append((v[z, y0:y1, x0:x1], f"t{t} z{z}"))
    if len(patches) >= 12: break
patches = patches[:12]
if patches:
    cols = 4; rowsn = int(np.ceil(len(patches)/cols))
    fig, ax = plt.subplots(rowsn, cols, figsize=(3*cols, 3*rowsn)); ax = np.array(ax).ravel()
    for i, (p, ttl) in enumerate(patches):
        ax[i].imshow(p, cmap="gray"); ax[i].set_title(ttl, fontsize=9)
        ax[i].axhline(p.shape[0]/2, color="lime", lw=0.5); ax[i].axvline(p.shape[1]/2, color="lime", lw=0.5)
        ax[i].axis("off")
    for j in range(len(patches), len(ax)): ax[j].axis("off")
    fig.suptitle(f"{SAMPLE} — GT merkezli {2*R}px yamalar (yeşil=merkez)")
    savefig("07_nucleus_patches.png")

In [ ]:
# GT konumlarindaki yogunluk vs rastgele voxel yogunlugu
gt_vals, rnd_vals = [], []
rng = np.random.RandomState(0)
for t in pick_t:
    v = vols[t]; sub = ndf[ndf.t == t] if "t" in ndf else ndf.iloc[:0]
    for _, r in sub.iterrows():
        z, y, x = int(round(r.z)), int(round(r.y)), int(round(r.x))
        z = min(max(z,0), v.shape[0]-1)
        gt_vals.append(float(v[z, min(max(y,0),Y-1), min(max(x,0),X-1)]))
    n = max(len(sub), 50)
    zz = rng.randint(0, v.shape[0], n); yy = rng.randint(0, Y, n); xx = rng.randint(0, X, n)
    rnd_vals.extend(v[zz, yy, xx].tolist())
if gt_vals:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].hist(rnd_vals, bins=60, alpha=0.6, density=True, label="rastgele")
    ax[0].hist(gt_vals, bins=30, alpha=0.6, density=True, label="GT konumu")
    ax[0].set_title("Yoğunluk: GT vs rastgele"); ax[0].legend(); ax[0].set_xlabel("intensite")
    ax[1].boxplot([rnd_vals, gt_vals], labels=["rastgele", "GT"]); ax[1].set_title("Yoğunluk kutu grafiği")
    savefig("07b_gt_vs_random_intensity.png")
    print("medyan yogunluk -> GT:%.1f  rastgele:%.1f" % (np.median(gt_vals), np.median(rnd_vals)))

## 8 · Soy yörüngesi ve görüntü üstüne bindirme

In [ ]:
# En uzun soyu sec, yorungeyi ciz.
comp_of = {int(i): find(int(i)) for i in gd["node_ids"]}
big = Counter(comp_of.values()).most_common(1)[0][0]
ln = ndf[ndf.id.map(lambda i: comp_of.get(int(i)) == big)].sort_values("t") if "t" in ndf else ndf
fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].imshow(vol.max(0), cmap="gray", vmax=np.percentile(vol,99.5))
if len(ln):
    ax[0].plot(ln.x, ln.y, "-", color="cyan", lw=1)
    sc = ax[0].scatter(ln.x, ln.y, c=ln.t, cmap="autumn", s=18)
    plt.colorbar(sc, ax=ax[0], fraction=0.04, label="t")
ax[0].set_title(f"En uzun soy yörüngesi (XY, n={len(ln)})"); ax[0].axis("off")
# tum GT merkezleri (t_mid)
sel = ndf[ndf.t == t_mid] if "t" in ndf else ndf.iloc[:0]
ax[1].imshow(vol.max(0), cmap="gray", vmax=np.percentile(vol,99.5))
if len(sel):
    ax[1].scatter(sel.x, sel.y, s=40, edgecolor="lime", facecolor="none", lw=1.2)
ax[1].set_title(f"t={t_mid} GT merkezleri (n={len(sel)})"); ax[1].axis("off")
savefig("08_lineage_and_overlay.png")

## 9 · Örnek gönderim (sample_submission.csv)

In [ ]:
ssub = ROOT/"sample_submission.csv" if ROOT else None
if ssub and ssub.exists():
    sdf = pd.read_csv(ssub)
    print("shape:", sdf.shape, "| kolonlar:", list(sdf.columns))
    print(sdf.head(10).to_string(index=False))
    print("\ndtypes:\n", sdf.dtypes)
else:
    print("sample_submission.csv bulunamadi")

## 10 · Bulgular (çalıştırdıktan sonra doldur)

Görselleri Kaggle Output'tan indirip `outputs/figures/`'a, bu notu `docs/`'a taşıyacağız.

**Dataset**
- [ ] train / test örnek sayısı = **… / …**
- [ ] Görüntü boyutu (T,Z,Y,X) aralığı = **…**, dtype = **…**, ölçek = **…**

**Ground-truth (seyreklik)**
- [ ] Örnek başına ortalama node / soy / bölünme = **…**
- [ ] Zaman başına etiketli node ≈ **…** (ne kadar seyrek?)
- [ ] **Koordinat birimi (COORDS_UM) = …** ← linking mesafe hesabı

**Hareket / linking**
- [ ] Kareler-arası yer değiştirme medyanı = **… µm**, 95p = **… µm** (7 µm toleransa göre arama yarıçapı)
- [ ] Hareket eksenler arası izotropik mi? (dz vs dy vs dx) = **…**

**Görüntü / detection**
- [ ] Z-derinliği boyunca yoğunluk zayıflaması var mı? = **…**
- [ ] GT konumlarındaki yoğunluk rastgeleden belirgin yüksek mi? = **…** (detection sinyali)

**Sonraki adım (Hafta 2):** `notebooks/02_baseline_ultrack.ipynb` — Ultrack ile ilk gönderim.